In [1]:
import sys, numpy as np, pandas as pd, statsmodels.api as sm
from statsmodels.stats.multitest import multipletests
sys.path.insert(0, '../../../scripts')
from score_sentiment import assign_trading_date

DATA = '../../../outputs/v3/data'
HOLDOUT_START = pd.Timestamp('2026-01-01')
sectors = pd.read_csv(f'{DATA}/sp500_tickers.csv').set_index('Symbol')['GICS Sector']
in_index = pd.read_csv(f'{DATA}/sp500_membership.csv', index_col=0, parse_dates=True).stack()

# 1. Daily sector-relative returns, index members only
px = pd.read_csv(f'{DATA}/features_df.csv', usecols=['Date', 'Ticker', 'ret_1d', 'ret_20d', 'vol_20d'],
                 index_col=['Date', 'Ticker'], parse_dates=['Date'])
px = px[in_index.reindex(px.index, fill_value=False).values]
sec = px.index.get_level_values('Ticker').map(sectors)
px['excess'] = px['ret_1d'] - px.groupby([px.index.get_level_values('Date'), sec])['ret_1d'].transform('median')
ex = px['excess'].unstack('Ticker').sort_index()          # Date x Ticker

# 2. Window returns: sum of excess returns from day D+a to D+b
cum = ex.fillna(0).cumsum()
def window(a, b):
    return (cum.shift(-b) - cum.shift(-(a - 1))).where(ex.notna()).stack()
car = pd.DataFrame({'ann': window(0, 1), 'drift': window(2, 21)})
td = ex.index
window_end = pd.Series(td[np.minimum(np.arange(len(td)) + 21, len(td) - 1)], index=td)

# 3. One row per release, with the change since the previous release
f = pd.read_csv(f'{DATA}/filing_signals_df.csv')
f['Date'] = assign_trading_date(f['Accepted'], td)
f = f.dropna(subset=['Date']).sort_values(['Ticker', 'Date']).drop_duplicates(['Ticker', 'Date'], keep='last')
signals = ['ai_demand_net', 'capex_net', 'ai_tone', 'ai_sentence_ratio']
for s in signals:
    f[f'd_{s}'] = f.groupby('Ticker')[s].diff()

ev = f.set_index(['Date', 'Ticker']).join(car, how='inner').join(px[['ret_20d', 'vol_20d']], how='inner')
ev = ev[window_end.reindex(ev.index.get_level_values('Date')).values < HOLDOUT_START]   # 2026 stays out
ev['quarter'] = ev.index.get_level_values('Date').to_period('Q')

# 4. Fama-MacBeth by quarter
controls = ['ann', 'ret_20d', 'vol_20d']
def fama_macbeth(signal):
    slopes = []
    for q, g in ev.dropna(subset=[signal]).groupby('quarter'):
        r = g[['drift', signal] + controls].rank(pct=True)
        slopes.append(sm.OLS(r['drift'], sm.add_constant(r[[signal] + controls])).fit().params[signal])
    s = pd.Series(slopes)
    m = sm.OLS(s.values, np.ones(len(s))).fit(cov_type='HAC', cov_kwds={'maxlags': 2})
    return s.mean(), m.tvalues[0], m.pvalues[0], len(s)

res = pd.DataFrame([fama_macbeth(f'd_{s}') for s in signals], index=signals,
                   columns=['slope', 't', 'p', 'quarters'])
res['p_holm'] = multipletests(res['p'], method='holm')[1]
res

,slope,t,p,quarters,p_holm
ai_demand_net,-0.076334,-1.415469,0.156931,28,0.470793
capex_net,-0.035601,-1.922344,0.054562,28,0.218250
ai_tone,0.003992,0.152552,0.878752,28,0.878752
ai_sentence_ratio,-0.017859,-0.890140,0.373391,28,0.746781
